# Proptech Engineer – Advanced Restaurant Invoice Tool

This notebook processes restaurant order data to generate styled Excel reports and professional PDF invoices. It includes functionalities for:
- Grouping restaurant branches (e.g., Sadia's Kitchen, Kheerwala).
- Generating styled Excel output suitable for A4 printing.
- Creating professional PDF invoices with automatic remarks and calculations.
- Producing a comprehensive operational report in PDF format with visual analytics.

In [ ]:
# =====================================================
#  Proptech Engineer – Advanced Restaurant Invoice Tool
#  - Groups branches (Sadia's Kitchen, Kheerwala, etc.)
#  - Styled Excel output ready for A4 printing
#  - Professional PDF invoices
# =====================================================

# !pip install fpdf2 openpyxl pandas --quiet

import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from google.colab import drive, files
from fpdf import FPDF
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# # Mount Google Drive
# drive.mount('/content/drive')


In [ ]:
# Install necessary libraries
!pip install fpdf2 openpyxl pandas num2words matplotlib --quiet

# Standard Library Imports
import re
import os
import math
from datetime import datetime, timedelta

# Third-party Library Imports
import pandas as pd
import numpy as np
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from num2words import num2words
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt

# Google Colab Specific Imports
from google.colab import drive, files

# Mount Google Drive (uncomment if needed)
# drive.mount('/content/drive')

## Helper Functions

This section contains all utility functions used throughout the notebook for data cleaning, grouping, text sanitization, calculations, and Excel styling.

In [ ]:
def clean_food(val):
    """Cleans and converts 'FoodValue' to a numeric format."""
    if pd.isna(val): return 0
    if isinstance(val, (int, float)): return val
    if isinstance(val, str):
        val = re.sub(r'[^¼-¾⅐-⅞+-/  -¯0-9¹²³µ÷½]', '', val) # Regex to allow numeric characters, fractions, and some symbols like +, -, /, ., comma to handle varied inputs properly.
        try: return float(val)
        except: return 0
    return 0

def get_group_name(original_name):
    """Groups restaurants based on custom rules and prefixes."""
    original = str(original_name).strip()
    # Special merges
    if original.startswith('Kacchi Express') or original.startswith('Bashmoti'):
        return 'Kacchi Express & Bashmoti'
    if original.startswith('Wish Cafe') or original.startswith('Craving Cup'):
        return 'Wish Cafe & Craving Cup'
    # Prefix groups (merged)
    prefixes = [
        "Sultan's Dine", "Sadia's Kitchen", "Kheerwala", "Town Chicken",
        "Sorisha Bari", "Fat Wrap", "VR Chittagong", "Lahori", "Lezzetli", "Subwala"
    ]
    for p in prefixes:
        if original.startswith(p):
            return f"{p} (All Branches)"
    # Keep original for others (Gapush Gupush, Hunger Killer, etc.)
    return original

def sanitize_text(text):
    """Sanitizes text for PDF compatibility by replacing special characters."""
    if not isinstance(text, str):
        text = str(text)
    replacements = {"’": "'", "“": '"', "”": '"', "–": "-", "—": "-", "…": "..."}
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text

def number_to_words(n):
    """Converts an integer to its English word representation."""
    return num2words(n, lang='en', to='cardinal').title()

def generate_remark(group_name, orders_df):
    """Generates custom remarks based on restaurant group rules or external CSV data."""
    if "Sultan's Dine" in group_name:
        nasirabad = orders_df[orders_df['Restaurant'].str.contains("02 No.", na=False)].shape[0]
        agrabad = orders_df[orders_df['Restaurant'].str.contains("Agrabad", na=False)].shape[0]
        return f"Nasirabad branch {nasirabad} order & Agrabad branch {agrabad} order"
    elif "Gapush Gupush" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 250].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 250/- Taka is {low_count}"
    elif "Sadia's Kitchen" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 300/- Taka is {low_count}"
    elif "Town Chicken" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        return f"The total number orders below 300/- Taka is {low_count:02d}"
    elif group_name == "Kacchi Express & Bashmoti":
        return "Added Bashmoti Restaurant"
    elif group_name == "Wish Cafe & Craving Cup":
        return "Added Craving Cup Restaurant"
    else:
        # Check if remark_df is available globally and not empty
        if 'remark_df' in globals() and remark_df is not None and not remark_df.empty:
            rem_row = remark_df[remark_df['restaurant_name'] == group_name]
            if not rem_row.empty:
                return rem_row.iloc[0]['remark']
        return ""

def calculate_monthly_charge(group_name, orders_df):
    """Calculates the monthly charge based on restaurant group and order count/value."""
    total_orders = len(orders_df)
    if group_name == "Kheerwala (All Branches)": return 1500
    if group_name == "Lahori (All Branches)": return 1000
    if group_name == "Kacchi Express & Bashmoti": return math.ceil(total_orders * 20)
    if group_name == "Wish Cafe & Craving Cup": return math.ceil(total_orders * 30)
    if group_name == "VR Chittagong (All Branches)": return math.ceil(total_orders * 25)
    if group_name == "Ladhidh (All Branches)": return math.ceil(total_orders * 25)
    if group_name == "Sorisha Bari (All Branches)": return math.ceil(total_orders * 20)
    if group_name == "Fat Wrap (All Branches)": return math.ceil(total_orders * 25)
    if "Sultan's Dine" in group_name: return math.ceil(total_orders * 26.25)
    if "Gapush Gupush" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 250].shape[0]
        return math.ceil(eligible * 25)
    if "Sadia's Kitchen" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 300].shape[0]
        return math.ceil(eligible * 35)
    if "Town Chicken" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 300].shape[0]
        return math.ceil(eligible * 35)
    if "Hunger Killer" in group_name: return math.ceil(total_orders * 25)
    return math.ceil(total_orders * 20)

def apply_excel_styling(worksheet, group_name, df_sheet, month_year):
    """Applies custom styling to an Excel worksheet, including headers and formatting."""
    # Insert title row with yellow background
    worksheet.insert_rows(1)
    title_cell = worksheet.cell(row=1, column=1)
    title_cell.value = f"{group_name} – {month_year}"
    title_cell.font = Font(bold=True, size=13, color="000000")
    title_cell.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(df_sheet.columns))
    worksheet.row_dimensions[1].height = 20

    # Header row (row 2)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin_border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )

    for col_idx, col_name in enumerate(df_sheet.columns, 1):
        cell = worksheet.cell(row=2, column=col_idx)
        cell.value = col_name
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border
        max_len = max(
            len(str(col_name)),
            df_sheet[col_name].astype(str).map(len).max() if len(df_sheet) > 0 else 10
        )
        worksheet.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 3, 35)

    # Data rows
    for row_idx, (_, row_data) in enumerate(df_sheet.iterrows(), start=3):
        for col_idx, col_name in enumerate(df_sheet.columns, 1):
            cell = worksheet.cell(row=row_idx, column=col_idx)
            cell.value = row_data[col_name]
            cell.border = thin_border
            cell.alignment = Alignment(horizontal="left", vertical="center")
            if (row_idx - 3) % 2 == 1:
                cell.fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
            if col_name == 'Date':
                try:
                    dt = pd.to_datetime(cell.value)
                    cell.value = dt.strftime('%d/%m/%Y')
                    cell.alignment = Alignment(horizontal="center")
                except:
                    pass
            if col_name == 'Status':
                cell.alignment = Alignment(horizontal="center")
            if col_name == 'FoodBill':
                cell.alignment = Alignment(horizontal="right")
                if isinstance(cell.value, (int, float)):
                    cell.value = f"{cell.value:,.0f}"

    worksheet.freeze_panes = 'A3'
    # No auto-filter
    ws_page = worksheet.page_setup
    ws_page.orientation = 'landscape'
    ws_page.paperSize = 9
    ws_page.fitToPage = True
    ws_page.fitToWidth = 1
    ws_page.fitToHeight = 0

In [ ]:
# -------------------------------------------------
# 1. Load order data
# -------------------------------------------------
print("="*60)
order_file = input("Full path to 'order_may.xlsx' in Drive: ").strip()

df = pd.read_excel(order_file, sheet_name=0, header=1)
df.columns = ['SL', 'Date', 'Unnamed', 'Restaurant', 'Rider', 'DC', 'Location',
              'Name', 'Number', 'FoodValue', 'OrderType', 'Commission', 'Feedback', 'DeliveryTime']
df = df[['Date', 'Restaurant', 'Rider', 'DC', 'Location', 'Name', 'Number', 'FoodValue', 'OrderType']].copy()
df = df.dropna(subset=['Restaurant'])
df = df[df['Restaurant'].str.strip() != '']

def clean_food(val):
    if pd.isna(val): return 0
    if isinstance(val, (int, float)): return val
    if isinstance(val, str):
        val = re.sub(r'[^\d.]', '', val)
        try: return float(val)
        except: return 0
    return 0

df['FoodValueNum'] = df['FoodValue'].apply(clean_food)
df['Date'] = pd.to_datetime(df['Date'])

Full path to 'order_may.xlsx' in Drive: /content/order_may.xlsx


In [ ]:
# -------------------------------------------------
# 2b. Group restaurants with custom merges and separate branches
# -------------------------------------------------
def get_group_name(original_name):
    original = str(original_name).strip()
    # Special merges
    if original.startswith('Kacchi Express') or original.startswith('Bashmoti'):
        return 'Kacchi Express & Bashmoti'
    if original.startswith('Wish Cafe') or original.startswith('Craving Cup'):
        return 'Wish Cafe & Craving Cup'
    # Prefix groups (merged)
    prefixes = [
        "Sultan's Dine", "Sadia's Kitchen", "Kheerwala", "Town Chicken",
        "Sorisha Bari", "Fat Wrap", "VR Chittagong", "Lahori", "Lezzetli", "Subwala"
    ]
    for p in prefixes:
        if original.startswith(p):
            return f"{p} (All Branches)"
    # Keep original for others (Gapush Gupush, Hunger Killer, etc.)
    return original

df['RestaurantGroup'] = df['Restaurant'].apply(get_group_name)

In [ ]:
# -------------------------------------------------
# 3. Create styled Excel file (same as before, with updated grouping)
# -------------------------------------------------
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

output_excel = "restaurant_orders_styled.xlsx"
month_year = df['Date'].dt.strftime('%B %Y').iloc[0]

def apply_excel_styling(worksheet, group_name, df_sheet):
    # Insert title row with yellow background
    worksheet.insert_rows(1)
    title_cell = worksheet.cell(row=1, column=1)
    title_cell.value = f"{group_name} – {month_year}"
    title_cell.font = Font(bold=True, size=13, color="000000")
    title_cell.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(df_sheet.columns))
    worksheet.row_dimensions[1].height = 20

    # Header row (row 2)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin_border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )

    for col_idx, col_name in enumerate(df_sheet.columns, 1):
        cell = worksheet.cell(row=2, column=col_idx)
        cell.value = col_name
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border
        max_len = max(
            len(str(col_name)),
            df_sheet[col_name].astype(str).map(len).max() if len(df_sheet) > 0 else 10
        )
        worksheet.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 3, 35)

    # Data rows
    for row_idx, (_, row_data) in enumerate(df_sheet.iterrows(), start=3):
        for col_idx, col_name in enumerate(df_sheet.columns, 1):
            cell = worksheet.cell(row=row_idx, column=col_idx)
            cell.value = row_data[col_name]
            cell.border = thin_border
            cell.alignment = Alignment(horizontal="left", vertical="center")
            if (row_idx - 3) % 2 == 1:
                cell.fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
            if col_name == 'Date':
                try:
                    dt = pd.to_datetime(cell.value)
                    cell.value = dt.strftime('%d/%m/%Y')
                    cell.alignment = Alignment(horizontal="center")
                except:
                    pass
            if col_name == 'Status':
                cell.alignment = Alignment(horizontal="center")
            if col_name == 'FoodBill':
                cell.alignment = Alignment(horizontal="right")
                if isinstance(cell.value, (int, float)):
                    cell.value = f"{cell.value:,.0f}"

    worksheet.freeze_panes = 'A3'
    # No auto-filter
    ws_page = worksheet.page_setup
    ws_page.orientation = 'landscape'
    ws_page.paperSize = 9
    ws_page.fitToPage = True
    ws_page.fitToWidth = 1
    ws_page.fitToHeight = 0

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    groups = df['RestaurantGroup'].unique()
    for group in groups:
        group_df = df[df['RestaurantGroup'] == group].copy()
        # Sort by original branch, then date
        group_df = group_df.sort_values(['Restaurant', 'Date'])
        group_df.reset_index(drop=True, inplace=True)
        group_df.insert(0, 'SL', group_df.index + 1)
        group_df = group_df.rename(columns={'FoodValue': 'FoodBill', 'OrderType': 'Status'})
        final_cols = ['SL', 'Date', 'Restaurant', 'Rider', 'DC', 'Location', 'Name', 'Number', 'FoodBill', 'Status']
        group_df = group_df[final_cols]
        sheet_name = group[:31]
        group_df.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        apply_excel_styling(worksheet, group, group_df)
        print(f"✓ Sheet '{sheet_name}' with {len(group_df)} orders")

print(f"\n✅ Styled Excel file: {output_excel}")

✓ Sheet 'Sultan's Dine (All Branches)' with 162 orders
✓ Sheet 'Gapush Gupush' with 74 orders
✓ Sheet 'Sadia's Kitchen (All Branches)' with 22 orders
✓ Sheet 'Lahori (All Branches)' with 7 orders
✓ Sheet 'Hunger Killer' with 13 orders
✓ Sheet 'Kheerwala (All Branches)' with 35 orders
✓ Sheet 'Gapush Gupush Chw' with 11 orders
✓ Sheet 'Town Chicken (All Branches)' with 9 orders
✓ Sheet 'Kacchi Express & Bashmoti' with 39 orders
✓ Sheet 'Hunger Killer 02 No.' with 10 orders
✓ Sheet 'Wish Cafe & Craving Cup' with 16 orders
✓ Sheet 'Subwala (All Branches)' with 7 orders
✓ Sheet 'Fat Wrap (All Branches)' with 22 orders
✓ Sheet 'Lezzetli (All Branches)' with 8 orders
✓ Sheet 'VR Chittagong (All Branches)' with 5 orders
✓ Sheet 'Sorisha Bari (All Branches)' with 22 orders

✅ Styled Excel file: restaurant_orders_styled.xlsx


In [ ]:
# -------------------------------------------------
# 4. Load address and remark CSV (as before)
# -------------------------------------------------
from google.colab import files

print("\n📁 Upload address.csv (columns: restaurant_name, address)")
addr_upload = files.upload()
addr_path = list(addr_upload.keys())[0]
address_df = pd.read_csv(addr_path)
address_df['restaurant_name'] = address_df['restaurant_name'].str.strip()

print("\n📁 Upload remark.csv (columns: restaurant_name, remark)")
rem_upload = files.upload()
rem_path = list(rem_upload.keys())[0]
remark_df = pd.read_csv(rem_path)
remark_df['restaurant_name'] = remark_df['restaurant_name'].str.strip()


📁 Upload address.csv (columns: restaurant_name, address)


Saving address.csv to address (2).csv

📁 Upload remark.csv (columns: restaurant_name, remark)


Saving remark.csv to remark (1).csv


In [ ]:
# -------------------------------------------------
# 5. Billing period and invoice date
# -------------------------------------------------
min_date = df['Date'].min()
max_date = df['Date'].max()
billing_start = min_date.strftime('%d/%m/%Y')
billing_end = max_date.strftime('%d/%m/%Y')
billing_period = f"From {billing_start} to {billing_end}"

today = datetime.now()
invoice_date = datetime(today.year, today.month, 7)
invoice_date_str = invoice_date.strftime("%d’th %B %Y")
due_date = invoice_date + timedelta(days=1)
due_date_str = due_date.strftime("%d’th %B %Y")

In [ ]:
# -------------------------------------------------
# 6. Function to generate remark automatically
# -------------------------------------------------
def generate_remark(group_name, orders_df):
    # For Sultan's Dine: count orders from 02 No. and Agrabad
    if "Sultan's Dine" in group_name:
        nasirabad = orders_df[orders_df['Restaurant'].str.contains("02 No.", na=False)].shape[0]
        agrabad = orders_df[orders_df['Restaurant'].str.contains("Agrabad", na=False)].shape[0]
        return f"Nasirabad branch {nasirabad} order & Agrabad branch {agrabad} order"
    # For Gapush Gupush (any branch)
    elif "Gapush Gupush" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 250].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 250/- Taka is {low_count}"
    # For Sadia's Kitchen
    elif "Sadia's Kitchen" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 300/- Taka is {low_count}"
    # For Town Chicken
    elif "Town Chicken" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        return f"The total number orders below 300/- Taka is {low_count:02d}"
    # For Kacchi Express & Bashmoti
    elif group_name == "Kacchi Express & Bashmoti":
        return "Added Bashmoti Restaurant"
    # For Wish Cafe & Craving Cup
    elif group_name == "Wish Cafe & Craving Cup":
        return "Added Craving Cup Restaurant"
    # Otherwise, use remark from CSV if available, else empty
    else:
        # This will be overridden by CSV later; we return empty here
        return ""

In [ ]:
# -------------------------------------------------
# FINAL PDF – AUTOMATIC WORDS, NO SIGNATURE LINE
# -------------------------------------------------
!pip install num2words --quiet

from fpdf import FPDF
from fpdf.enums import XPos, YPos
from datetime import datetime, timedelta
import os, math
from num2words import num2words

# ---------- Helper functions ----------
def sanitize_text(text):
    if not isinstance(text, str):
        text = str(text)
    replacements = {"’": "'", "“": '"', "”": '"', "–": "-", "—": "-", "…": "..."}
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text

def number_to_words(n):
    """Convert integer to words using num2words."""
    return num2words(n, lang='en', to='cardinal').title()

def generate_remark(group_name, orders_df):
    if "Sultan's Dine" in group_name:
        nasirabad = orders_df[orders_df['Restaurant'].str.contains("02 No.", na=False)].shape[0]
        agrabad = orders_df[orders_df['Restaurant'].str.contains("Agrabad", na=False)].shape[0]
        return f"Nasirabad branch {nasirabad} order & Agrabad branch {agrabad} order"
    elif "Gapush Gupush" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 250].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 250/- Taka is {low_count}"
    elif "Sadia's Kitchen" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        total = len(orders_df)
        return f"Number of orders {total}. The total number orders below 300/- Taka is {low_count}"
    elif "Town Chicken" in group_name:
        low_count = orders_df[orders_df['FoodBill'] < 300].shape[0]
        return f"The total number orders below 300/- Taka is {low_count:02d}"
    elif group_name == "Kacchi Express & Bashmoti":
        return "Added Bashmoti Restaurant"
    elif group_name == "Wish Cafe & Craving Cup":
        return "Added Craving Cup Restaurant"
    else:
        if 'remark_df' in globals():
            rem_row = remark_df[remark_df['restaurant_name'] == group_name]
            if not rem_row.empty:
                return rem_row.iloc[0]['remark']
        return ""

def calculate_monthly_charge(group_name, orders_df):
    total_orders = len(orders_df)
    if group_name == "Kheerwala (All Branches)": return 1500
    if group_name == "Lahori (All Branches)": return 1000
    if group_name == "Kacchi Express & Bashmoti": return math.ceil(total_orders * 20)
    if group_name == "Wish Cafe & Craving Cup": return math.ceil(total_orders * 30)
    if group_name == "VR Chittagong (All Branches)": return math.ceil(total_orders * 25)
    if group_name == "Ladhidh (All Branches)": return math.ceil(total_orders * 25)
    if group_name == "Sorisha Bari (All Branches)": return math.ceil(total_orders * 20)
    if group_name == "Fat Wrap (All Branches)": return math.ceil(total_orders * 25)
    if "Sultan's Dine" in group_name: return math.ceil(total_orders * 26.25)
    if "Gapush Gupush" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 250].shape[0]
        return math.ceil(eligible * 25)
    if "Sadia's Kitchen" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 300].shape[0]
        return math.ceil(eligible * 35)
    if "Town Chicken" in group_name:
        eligible = orders_df[orders_df['FoodBill'] >= 300].shape[0]
        return math.ceil(eligible * 35)
    if "Hunger Killer" in group_name: return math.ceil(total_orders * 25)
    return math.ceil(total_orders * 20)

# ---------- PDF Class (no signature line, automatic words) ----------
class InvoicePDF(FPDF):
    def __init__(self, logo_path=None):
        super().__init__()
        self.logo_path = logo_path
        self.set_auto_page_break(auto=True, margin=25)

    def add_invoice_page(self, group_name, orders_df, address, monthly_charge,
                         billing_period, invoice_date_str, due_date_str):
        self.add_page()
        # Border
        self.set_draw_color(0, 0, 0)
        self.rect(8, 8, 194, 281, style='D')

        # Logo top right
        if self.logo_path and os.path.exists(self.logo_path):
            self.image(self.logo_path, x=165, y=15, w=30)
            title_y = 30
        else:
            title_y = 20

        # Title
        self.set_y(title_y)
        self.set_font('Helvetica', 'BU', 23)
        self.set_text_color(0, 0, 0)
        self.cell(0, 15, 'INVOICE', align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(6)

        # Invoice No & Date
        self.set_font('Helvetica', '', 11)
        inv_no = f"#FR{group_name[:3].upper()}{invoice_date.strftime('%m%y')}"
        self.cell(0, 7, f"Invoice No: {inv_no}", align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.cell(0, 7, f"Date: {sanitize_text(invoice_date_str)}", align='R', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(10)

        # Label-value pairs
        def label_value(label, value):
            self.set_font('Helvetica', 'BU', 12)
            self.cell(50, 9, label, align='L')
            self.set_font('Helvetica', '', 12)
            self.cell(0, 9, value, align='L', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
            self.ln(3)

        label_value("Restaurant Name:", sanitize_text(group_name))
        label_value("Branch:", sanitize_text(address))
        label_value("Billing Period:", sanitize_text(billing_period))

        remark = generate_remark(group_name, orders_df)
        if remark and remark != "nan":
            self.set_font('Helvetica', 'BU', 12)
            self.cell(50, 9, "Remarks:", align='L')
            self.set_font('Helvetica', '', 12)
            self.multi_cell(0, 7, sanitize_text(remark))
            self.ln(4)

        self.ln(6)

        # Totals
        total_orders = len(orders_df)
        total_food = orders_df['FoodBill'].sum()
        monthly_int = int(monthly_charge)
        self.set_font('Helvetica', 'B', 12)
        totals = f"Total Number of Orders: {total_orders}   Value of Order: {total_food:,.0f}/-   Monthly Recurring Charge: {monthly_int}/-"
        self.cell(0, 9, totals, align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(4)

        # In words – automatic
        words = number_to_words(monthly_int)
        self.set_font('Helvetica', 'I', 11)
        self.cell(0, 7, f"(In word: {words} Taka only)", align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(10)

        # Payment paragraph
        self.set_font('Helvetica', '', 11)
        self.set_text_color(0, 0, 0)
        self.write(7, "Please pay the amount within ")
        self.set_font('Helvetica', 'B', 11)
        self.write(7, sanitize_text(due_date_str))
        self.set_font('Helvetica', '', 11)
        self.write(7, " to keep the services going on. Payment can be made in cash or by ")
        self.set_font('Helvetica', 'B', 11)
        self.set_text_color(255, 105, 180)
        self.write(7, "bKash")
        self.set_font('Helvetica', '', 11)
        self.set_text_color(0, 0, 0)
        self.write(7, " Payment (018 94 94 35 23). We appreciate any suggestions to improve our services. For further query please feel free to contact with us in ")
        self.set_font('Helvetica', 'B', 11)
        self.write(7, "+88 018 94 94 35 30")
        self.set_font('Helvetica', '', 11)
        self.write(7, " or ")
        self.set_font('Helvetica', 'B', 11)
        self.set_text_color(0, 0, 255)
        self.write(7, "accounts@foodrush.xyz")
        self.set_font('Helvetica', '', 11)
        self.set_text_color(0, 0, 0)
        self.write(7, ".")
        self.ln(7)

        self.set_font('Helvetica', 'I', 11)
        self.write(7, "Thanks for staying with Foodrush.")
        self.ln(20)

        # Signatures at bottom
        y = self.get_y()
        page_h = 297
        margin_b = 30
        remaining = page_h - y - margin_b
        if remaining < 40:
            self.ln(remaining - 10)
        else:
            self.ln(remaining - 25)
        self.set_font('Helvetica', 'B', 11)
        self.cell(95, 9, "Executive-Accounts & Operations", align='C')
        self.cell(95, 9, "Executive-Sales and MKT", align='C')
        self.ln(9)
        self.set_font('Helvetica', '', 11)
        self.cell(95, 7, "Foodrush", align='C')
        self.cell(95, 7, "Foodrush", align='C')

# ---------- Prepare dates ----------
min_date = df['Date'].min()
max_date = df['Date'].max()
billing_start = min_date.strftime('%d/%m/%Y')
billing_end = max_date.strftime('%d/%m/%Y')
billing_period = f"From {billing_start} to {billing_end}"
today = datetime.now()
invoice_date = datetime(today.year, today.month, 7)
invoice_date_str = invoice_date.strftime("%d’th %B %Y")
due_date = invoice_date + timedelta(days=1)
due_date_str = due_date.strftime("%d’th %B %Y")

# ---------- Generate PDF (fully automatic) ----------
logo_file = "logo.png"
if not os.path.exists(logo_file):
    print("⚠️ Logo not found. Proceeding without logo.")
    logo_file = None

pdf = InvoicePDF(logo_path=logo_file)
groups = df['RestaurantGroup'].unique()

for group in groups:
    print(f"Generating PDF for: {group}")
    orders_group = df[df['RestaurantGroup'] == group].copy()
    orders_group['FoodBill'] = orders_group['FoodValueNum']
    addr_row = address_df[address_df['restaurant_name'] == group]
    if addr_row.empty:
        sample = orders_group['Restaurant'].iloc[0] if len(orders_group) > 0 else group
        addr_row = address_df[address_df['restaurant_name'] == sample]
    address = addr_row.iloc[0]['address'] if not addr_row.empty else "Address not provided"
    monthly = calculate_monthly_charge(group, orders_group)

    pdf.add_invoice_page(group, orders_group, address, monthly, billing_period,
                         invoice_date_str, due_date_str)

pdf.output("All_Invoices_Automatic.pdf")
print("\n✅ PDF saved: All_Invoices_Automatic.pdf")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 10.2 MB/s eta 0:00:00
Generating PDF for: Sultan's Dine (All Branches)
Generating PDF for: Gapush Gupush
Generating PDF for: Sadia's Kitchen (All Branches)
Generating PDF for: Lahori (All Branches)
Generating PDF for: Hunger Killer
Generating PDF for: Kheerwala (All Branches)
Generating PDF for: Gapush Gupush Chw
Generating PDF for: Town Chicken (All Branches)
Generating PDF for: Kacchi Express & Bashmoti
Generating PDF for: Hunger Killer 02 No.
Generating PDF for: Wish Cafe & Craving Cup
Generating PDF for: Subwala (All Branches)
Generating PDF for: Fat Wrap (All Branches)
Generating PDF for: Lezzetli (All Branches)
Generating PDF for: VR Chittagong (All Branches)
Generating PDF for: Sorisha Bari (All Branches)

✅ PDF saved: All_Invoices_Automatic.pdf


In [ ]:
# -------------------------------------------------
# PROFESSIONAL PDF REPORT – FULLY CORRECTED
# -------------------------------------------------
!pip install matplotlib --quiet
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
from fpdf import FPDF
from fpdf.enums import XPos, YPos
import os
import math

# ---------- Helper: sanitize text ----------
def sanitize_text(text):
    if not isinstance(text, str):
        text = str(text)
    replacements = {
        "’": "'", "“": '"', "”": '"',
        "–": "-", "—": "-", "…": "...",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text

# Ensure necessary columns
df['FoodBill'] = df['FoodValueNum']
df['DateOnly'] = df['Date'].dt.date

# Group summaries
group_summary = df.groupby('RestaurantGroup').agg(
    Total_Orders=('Restaurant', 'count'),
    Total_Food_Value=('FoodBill', 'sum')
).reset_index()
group_summary['Monthly_Charge'] = group_summary['RestaurantGroup'].apply(
    lambda g: calculate_monthly_charge(g, df[df['RestaurantGroup'] == g])
)
group_summary = group_summary.sort_values('Monthly_Charge', ascending=False)

# Branch summary
branch_summary = df.groupby('Restaurant').agg(
    Orders=('Restaurant', 'count'),
    Food_Value=('FoodBill', 'sum')
).reset_index().sort_values('Orders', ascending=False).head(20)

# Overall totals
total_orders = df.shape[0]
total_food = df['FoodBill'].sum()
total_monthly = group_summary['Monthly_Charge'].sum()
avg_order = total_food / total_orders if total_orders > 0 else 0

# Daily orders
daily_orders = df.groupby('DateOnly').size().reset_index(name='Orders')
daily_orders = daily_orders.sort_values('DateOnly')

# Create charts
# 1. Bar chart
top_groups = group_summary.head(10)
plt.figure(figsize=(8, 4))
plt.bar(top_groups['RestaurantGroup'], top_groups['Monthly_Charge'], color='steelblue')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('Monthly Charge (Tk)')
plt.title('Top 10 Restaurant Groups by Monthly Recurring Charge')
plt.tight_layout()
chart1_path = 'chart1.png'
plt.savefig(chart1_path, dpi=150)
plt.close()

# 2. Pie chart
pie_data = group_summary.copy()
pie_data['RestaurantGroup'] = pie_data['RestaurantGroup'].apply(lambda x: x[:25] + '..' if len(x) > 25 else x)
other_sum = pie_data.iloc[5:]['Monthly_Charge'].sum()
top5 = pie_data.head(5)
if other_sum > 0:
    top5 = pd.concat([top5, pd.DataFrame([{'RestaurantGroup': 'Others', 'Monthly_Charge': other_sum}])])
plt.figure(figsize=(6, 5))
plt.pie(top5['Monthly_Charge'], labels=top5['RestaurantGroup'], autopct='%1.1f%%', startangle=90)
plt.title('Monthly Charge Distribution (by Group)')
plt.tight_layout()
chart2_path = 'chart2.png'
plt.savefig(chart2_path, dpi=150)
plt.close()

# 3. Line chart
plt.figure(figsize=(8, 3))
plt.plot(daily_orders['DateOnly'], daily_orders['Orders'], marker='o', linestyle='-', color='green', markersize=3)
plt.xlabel('Date')
plt.ylabel('Number of Orders')
plt.title('Daily Order Volume (May 2026)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
chart3_path = 'chart3.png'
plt.savefig(chart3_path, dpi=150)
plt.close()

# ---------- PDF class ----------
class ReportPDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 10, sanitize_text('Foodrush – Operational Report (May 2026)'), 0, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')

    def chapter_title(self, title):
        self.set_font('Helvetica', 'B', 14)
        self.set_fill_color(200, 220, 255)
        self.cell(0, 10, sanitize_text(title), 0, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='L', fill=1)
        self.ln(4)

    def chapter_body(self, body):
        self.set_font('Helvetica', '', 11)
        self.multi_cell(0, 6, sanitize_text(body))
        self.ln()

    def add_table(self, headers, data, col_widths=None):
        if not col_widths:
            col_widths = [40, 30, 40, 40]
        self.set_font('Helvetica', 'B', 10)
        for i, header in enumerate(headers):
            self.cell(col_widths[i], 8, sanitize_text(header), 1, 0, 'C')
        self.ln()
        self.set_font('Helvetica', '', 10)
        for row in data:
            for i, item in enumerate(row):
                self.cell(col_widths[i], 7, sanitize_text(str(item)), 1, 0, 'L')
            self.ln()

# ---------- Build PDF ----------
pdf = ReportPDF()
pdf.add_page()

# Page 1: Executive Summary
pdf.chapter_title("1. Executive Summary")
summary_text = (f"Total Number of Orders: {total_orders:,}\n"
                f"Total Food Value: {total_food:,.0f} Tk\n"
                f"Total Monthly Recurring Charges: {total_monthly:,.0f} Tk\n"
                f"Average Order Value: {avg_order:,.2f} Tk\n"
                f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}\n"
                f"Number of Active Restaurants (groups): {group_summary.shape[0]}\n"
                f"Number of Branches: {df['Restaurant'].nunique()}")
pdf.chapter_body(summary_text)

# Add bar chart
pdf.image(chart1_path, x=10, y=pdf.get_y(), w=190)
pdf.set_y(pdf.get_y() + 60)
pdf.ln(10)

# Page 2: Detailed Tables
pdf.add_page()
pdf.chapter_title("2. Restaurant Group Performance")
headers = ['Restaurant Group', 'Orders', 'Food Value (Tk)', 'Monthly Charge (Tk)']
data = []
for _, row in group_summary.iterrows():
    data.append([row['RestaurantGroup'][:35], row['Total_Orders'], f"{row['Total_Food_Value']:,.0f}", f"{row['Monthly_Charge']:,.0f}"])
col_widths = [60, 25, 50, 50]
pdf.add_table(headers, data, col_widths)
pdf.ln(10)

pdf.chapter_title("3. Branch Breakdown (Top 20)")
headers2 = ['Branch Name', 'Orders', 'Food Value (Tk)']
data2 = []
for _, row in branch_summary.iterrows():
    data2.append([row['Restaurant'][:40], row['Orders'], f"{row['Food_Value']:,.0f}"])
col_widths2 = [100, 40, 50]
pdf.add_table(headers2, data2, col_widths2)

# Page 3: Charts and Additional Stats
pdf.add_page()
pdf.chapter_title("4. Visual Analytics")
# Pie chart
pdf.image(chart2_path, x=10, y=pdf.get_y(), w=90)
# Daily orders chart
pdf.image(chart3_path, x=100, y=pdf.get_y() - 20, w=100)
pdf.set_y(pdf.get_y() + 80)
pdf.ln(10)

pdf.chapter_title("5. Additional Metrics")
extra_text = (f"Orders with zero food value (PAID): {(df['FoodBill'] == 0).sum()}\n"
              f"Highest single order: {df['FoodBill'].max():,.0f} Tk\n"
              f"Busiest day: {daily_orders.loc[daily_orders['Orders'].idxmax(), 'DateOnly']} "
              f"({daily_orders['Orders'].max()} orders)\n")
pdf.chapter_body(extra_text)

# Remarks table (if exists, handle NaN)
if 'remark_df' in globals() and remark_df is not None and not remark_df.empty:
    pdf.chapter_title("6. Remarks Summary")
    remarks_data = []
    for _, row in remark_df.iterrows():
        rest_name = sanitize_text(row['restaurant_name'])
        remark_val = row['remark']
        # Convert to string and handle NaN
        if pd.isna(remark_val):
            remark_str = ""
        else:
            remark_str = sanitize_text(str(remark_val))[:60]
        remarks_data.append([rest_name, remark_str])
    pdf.add_table(['Restaurant', 'Remark'], remarks_data, [70, 120])

# Output PDF
pdf.output("Foodrush_Operational_Report.pdf")
print("\n✅ PDF report saved: Foodrush_Operational_Report.pdf")

# Clean up temporary chart images
for f in [chart1_path, chart2_path, chart3_path]:
    if os.path.exists(f):
        os.remove(f)


✅ PDF report saved: Foodrush_Operational_Report.pdf


/tmp/ipykernel_36064/1041029893.py:123: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(col_widths[i], 8, sanitize_text(header), 1, 0, 'C')
/tmp/ipykernel_36064/1041029893.py:128: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(col_widths[i], 7, sanitize_text(str(item)), 1, 0, 'L')
/tmp/ipykernel_36064/1041029893.py:123: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(col_widths[i], 8, sanitize_text(header), 1, 0, 'C')
/tmp/ipykernel_36064/1041029893.py:128: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(col_widths[i], 7, sanitize_text(str(item)), 1, 0, 'L')
